In [ ]:
from google.colab import files
uploaded = files.upload()
# select orders.csv, city.csv, restaurants.csv, districts.csv,
# zone_distance_matrix.npy, simulation_config.json

TypeError: 'NoneType' object is not subscriptable

In [ ]:
import pandas as pd

orders = pd.read_csv("orders.csv")

print("Shape (rows, cols):", orders.shape)
print("\nColumn names:\n", orders.columns.tolist())
print("\nDtypes:\n", orders.dtypes)
print("\nNull counts:\n", orders.isnull().sum())
print("\nFirst 5 rows:\n", orders.head())

In [ ]:
import matplotlib.pyplot as plt

# timestamp_sec = seconds since midnight (e.g. 36000 = 10:00:00 AM)
timestamp_col = "timestamp_sec"

orders["hour"] = (orders[timestamp_col] // 3600) % 24

plt.figure(figsize=(10, 5))
plt.hist(orders["hour"], bins=24, range=(0, 24), edgecolor="black")
plt.xlabel("Hour of Day")
plt.ylabel("Number of Orders")
plt.title("Order Timestamp Distribution")
plt.xticks(range(0, 24))
plt.axvspan(12, 14, color="orange", alpha=0.2, label="Lunch (12-2pm)")
plt.axvspan(19, 22, color="purple", alpha=0.2, label="Dinner (7-10pm)")
plt.legend()
plt.show()

print(orders["hour"].value_counts().sort_index())

# Bonus cross-check: orders.csv already has a peak_window label — compare against your histogram
print("\npeak_window value counts:\n", orders["peak_window"].value_counts())

In [ ]:
import pandas as pd
import numpy as np
import json

restaurants = pd.read_csv("restaurants.csv")
districts = pd.read_csv("districts.csv")

with open("simulation_config.json") as f:
    sim_config = json.load(f)

zone_id_to_index = {zid: idx for idx, zid in enumerate(districts["district_id"])}

restaurants = restaurants.merge(
    districts[["district_id", "centroid_lat", "centroid_lon"]],
    left_on="zone_id", right_on="district_id", how="left"
).rename(columns={"centroid_lat": "current_lat", "centroid_lon": "current_lon"})

restaurants["zone_index"] = restaurants["zone_id"].map(zone_id_to_index)
restaurant_zone_lookup = restaurants.set_index("restaurant_id")["zone_index"]

restaurants["status"] = "available"
restaurants["return_time"] = 0

partners = restaurants.to_dict("records")

NORMAL_ACTIVE_PARTNERS = sim_config["active_partners_normal"]
PEAK_ACTIVE_PARTNERS = sim_config["active_partners_peak"]
NIGHT_ACTIVE_PARTNERS = sim_config["active_partners_night"]

PEAK_HOURS = {12, 13, 19, 20, 21}
NIGHT_HOURS = {22, 23}

HOURLY_ACTIVE_TARGET = {
    h: (NIGHT_ACTIVE_PARTNERS if h in NIGHT_HOURS
        else PEAK_ACTIVE_PARTNERS if h in PEAK_HOURS
        else NORMAL_ACTIVE_PARTNERS)
    for h in range(10, 24)
}

print(f"Loaded {len(partners)} total restaurants")
print(f"Hourly active-partner targets: {HOURLY_ACTIVE_TARGET}")

In [ ]:
import numpy as np

zone_matrix = np.load("zone_distance_matrix.npy")

print("Shape:", zone_matrix.shape)
print("Is square:", zone_matrix.shape[0] == zone_matrix.shape[1])

# Cross-check against actual number of zones/districts
num_districts = districts.shape[0]
print(f"Number of districts in districts.csv: {num_districts}")
print(f"Matrix shape matches district count: {zone_matrix.shape[0] == num_districts}")

diagonal = np.diag(zone_matrix)
print("\nDiagonal all zero:", np.all(diagonal == 0))
print("Max diagonal value:", diagonal.max())

print("Any negative values:", (zone_matrix < 0).any())
print("Min value:", zone_matrix.min())
print("Max value:", zone_matrix.max())

In [ ]:
city = pd.read_csv("city.csv")

readme_content = f"""# Data Documentation

## orders.csv
Contains {orders.shape[0]} rows and {orders.shape[1]} columns: {', '.join(orders.columns)}.
Each row is a synthetic food delivery order with `timestamp_sec` (seconds since midnight),
prep time, SLA deadline, order value, and a `peak_window` label indicating lunch/dinner/off-peak.

## city.csv
Contains {city.shape[0]} row(s) and {city.shape[1]} columns: {', '.join(city.columns)}.
Stores network-level metadata for the Hyderabad OSMnx road network — reference point,
search radius, bounding box (bbox_north/south/east/west), node/edge counts, and the
associated GraphML file name. Not per-order location data.

## restaurants.csv
Contains {restaurants.shape[0]} rows and {restaurants.shape[1]} columns: {', '.join(restaurants.columns)}.
Represents active food delivery restaurants, each assigned to a `zone_id` (joins to districts.csv)
along with size tier, prep time, and popularity weight. Restaurant coordinates are derived
from their zone's centroid.

## districts.csv
Contains {districts.shape[0]} rows and {districts.shape[1]} columns: {', '.join(districts.columns)}.
Defines real Hyderabad neighborhood zones (hardcoded centroids, not pie-slice geometry), each with
centroid coordinates, demand weight, and area in km². Used for spatial zoning of orders and depots.

## zone_distance_matrix.npy
A {zone_matrix.shape[0]}x{zone_matrix.shape[1]} NumPy array of pairwise distances between zones/districts.
Diagonal values are 0 and all values are non-negative, used for routing calculations in the simulation.
"""

with open("README_data.md", "w") as f:
    f.write(readme_content)

print(readme_content)

In [ ]:
# from google.colab import files
# files.download("README_data.md")

Day 2

In [ ]:
import pandas as pd

orders = pd.read_csv("orders.csv")
districts = pd.read_csv("districts.csv")

# Join destination_zone_id to districts.csv to get delivery coordinates
orders_sorted = orders.merge(
    districts[["district_id", "centroid_lat", "centroid_lon"]],
    left_on="destination_zone_id",
    right_on="district_id",
    how="left"
).rename(columns={"centroid_lat": "delivery_lat", "centroid_lon": "delivery_lon"})

# Sort by timestamp ascending — this is the clean simulation input
orders_sorted = orders_sorted.sort_values("timestamp_sec").reset_index(drop=True)

print("Shape:", orders_sorted.shape)
print("\nSorted correctly:", orders_sorted["timestamp_sec"].is_monotonic_increasing)
print("\nFirst 5 rows:\n", orders_sorted[["order_id", "restaurant_id", "delivery_lat", "delivery_lon", "timestamp_sec"]].head())

In [ ]:
class OrderStream:
    """Feeds orders one at a time in timestamp order, simulating real-time arrival."""

    def __init__(self, sorted_orders_df):
        self.orders = sorted_orders_df.reset_index(drop=True)
        self.current_index = 0
        self.total_orders = len(self.orders)

    def next(self):
        """Return the next order as a dict, or None if stream is exhausted."""
        if self.current_index >= self.total_orders:
            return None
        order = self.orders.iloc[self.current_index].to_dict()
        self.current_index += 1
        return order

    def has_next(self):
        return self.current_index < self.total_orders

    def reset(self):
        self.current_index = 0


# Instantiate the stream with our sorted data
stream = OrderStream(orders_sorted)
print(f"OrderStream initialized with {stream.total_orders} orders")

In [ ]:
# sim_clock tracks simulation time — updates to each order's timestamp as it's fed
sim_clock = None

def advance_clock(order):
    global sim_clock
    sim_clock = order["timestamp_sec"]
    return sim_clock

stream.reset()
first_order = stream.next()
advance_clock(first_order)
print(f"sim_clock after first order: {sim_clock} (should equal {first_order['timestamp_sec']})")

stream.reset()
sim_clock = None

In [ ]:
stream.reset()
sim_clock = None

print(f"{'order_id':<12}{'restaurant_id':<16}{'delivery_lat':<15}{'delivery_lon':<15}{'timestamp_sec':<15}")
print("-" * 73)

for i in range(50):
    order = stream.next()
    if order is None:
        print("Stream exhausted early.")
        break
    advance_clock(order)
    print(f"{order['order_id']:<12}{order['restaurant_id']:<16}"
          f"{order['delivery_lat']:<15.4f}{order['delivery_lon']:<15.4f}{order['timestamp_sec']:<15}")

print(f"\nFinal sim_clock: {sim_clock}")
print(f"Orders consumed: {stream.current_index}")

In [ ]:
# Compute hour of day from timestamp_sec, then flag lunch (12-14h) or dinner (19-22h)
orders_sorted["hour"] = (orders_sorted["timestamp_sec"] // 3600) % 24
orders_sorted["peak_hour"] = orders_sorted["hour"].apply(
    lambda h: (12 <= h < 14) or (19 <= h < 22)
)

print(orders_sorted["peak_hour"].value_counts())
print("\nSample rows:\n", orders_sorted[["order_id", "timestamp_sec", "hour", "peak_hour"]].head(10))

# Cross-check against orders.csv's existing peak_window label
print("\nCross-check against peak_window:\n",
      pd.crosstab(orders_sorted["peak_hour"], orders_sorted["peak_window"]))

Day 3

In [ ]:
import pandas as pd
import numpy as np
import json

restaurants = pd.read_csv("restaurants.csv")
districts = pd.read_csv("districts.csv")

with open("simulation_config.json") as f:
    sim_config = json.load(f)

ACTIVE_PARTNERS = sim_config["active_partners_normal"]

# Positional map: district_id -> index in zone_distance_matrix.npy
zone_id_to_index = {zid: idx for idx, zid in enumerate(districts["district_id"])}

# Attach each restaurant's current_location = its zone's centroid
restaurants = restaurants.merge(
    districts[["district_id", "centroid_lat", "centroid_lon"]],
    left_on="zone_id", right_on="district_id", how="left"
).rename(columns={"centroid_lat": "current_lat", "centroid_lon": "current_lon"})

restaurants["zone_index"] = restaurants["zone_id"].map(zone_id_to_index)

np.random.seed(42)
active_mask = np.zeros(len(restaurants), dtype=bool)
active_idx = np.random.choice(len(restaurants), size=min(ACTIVE_PARTNERS, len(restaurants)), replace=False)
active_mask[active_idx] = True

restaurants["status"] = np.where(active_mask, "available", "off_shift")
restaurants["return_time"] = 0

partners = restaurants.to_dict("records")

print(f"Loaded {len(partners)} total restaurants")
print()
print(pd.DataFrame(partners)[["restaurant_id", "zone_id", "zone_name", "current_lat", "current_lon", "status"]].head())

restaurant_zone_lookup = restaurants.set_index("restaurant_id")["zone_index"]

In [ ]:
def find_nearest_partner(order, partners, zone_matrix):
    """Return (partner, pickup_sec) for the closest AVAILABLE partner to the RESTAURANT
    (pickup point) — not the destination. The partner must reach the restaurant first."""
    restaurant_zone_idx = int(restaurant_zone_lookup[order["restaurant_id"]])

    nearest, nearest_time = None, float("inf")
    for p in partners:
        if p["status"] != "available":
            continue
        if p["zone_index"] == restaurant_zone_idx:
            pickup_sec = INTRA_ZONE_SEC
        else:
            pickup_sec = zone_matrix[p["zone_index"], restaurant_zone_idx]
        if pickup_sec < nearest_time:
            nearest, nearest_time = p, pickup_sec

    return nearest, nearest_time

In [ ]:
off_diag_mask = ~np.eye(zone_matrix.shape[0], dtype=bool)
avg_cross_zone_sec = zone_matrix[off_diag_mask].mean()

districts["zone_radius_km"] = np.sqrt(districts["area_km2"] / np.pi)
districts["intra_zone_avg_km"] = 0.905 * districts["zone_radius_km"]
DELIVERY_SPEED_KMH = 30
INTRA_ZONE_KM = districts["intra_zone_avg_km"].mean()
INTRA_ZONE_SEC = INTRA_ZONE_KM / DELIVERY_SPEED_KMH * 3600

PEAK_HOURS = {12, 13, 19, 20, 21}
NIGHT_HOURS_SET = {22, 23}

DEFAULT_CANCELLATION_SEC = 600
PEAK_CANCELLATION_SEC = 480
NIGHT_CANCELLATION_SEC = 240

def cancellation_window_for_hour(hour):
    if hour in NIGHT_HOURS_SET:
        return NIGHT_CANCELLATION_SEC
    elif hour in PEAK_HOURS:
        return PEAK_CANCELLATION_SEC
    else:
        return DEFAULT_CANCELLATION_SEC

def release_returned_partners(partners, current_time, target_active):
    """Free up finished partners, but cap how many rejoin 'available' at the
    current hour's target — a partner finishing mid-hour shouldn't silently
    exceed the shift size just because they happened to start earlier."""
    finishing = [p for p in partners if p["status"] == "busy" and p["return_time"] <= current_time]
    currently_active = sum(1 for p in partners if p["status"] in ("available", "busy")) - len(finishing)

    for p in finishing:
        if currently_active < target_active:
            p["status"] = "available"
            currently_active += 1
        else:
            p["status"] = "off_shift"

def rebalance_active_partners(partners, target_active, current_time):
    non_busy = [p for p in partners if p["status"] != "busy"]
    busy_count = len(partners) - len(non_busy)
    needed_available = min(max(target_active - busy_count, 0), len(non_busy))
    non_busy_sorted = sorted(non_busy, key=lambda p: p["restaurant_id"])
    rng = np.random.RandomState(int(current_time))
    rng.shuffle(non_busy_sorted)
    for i, p in enumerate(non_busy_sorted):
        p["status"] = "available" if i < needed_available else "off_shift"

def dispatch_greedy(order_stream, partners, zone_matrix, max_orders=None):
    dispatch_log = []
    pending_queue = []
    order_stream.reset()
    count = 0
    current_hour = None

    def try_dispatch(order, current_time):
        partner, pickup_sec = find_nearest_partner(order, partners, zone_matrix)
        if partner is None:
            return None
        restaurant_zone_idx = int(restaurant_zone_lookup[order["restaurant_id"]])
        dest_zone_idx = zone_id_to_index[order["destination_zone_id"]]
        dropoff_sec = INTRA_ZONE_SEC if restaurant_zone_idx == dest_zone_idx \
            else zone_matrix[restaurant_zone_idx, dest_zone_idx]
        total_travel_sec = pickup_sec + dropoff_sec

        partner["status"] = "busy"
        partner["return_time"] = current_time + total_travel_sec
        partner["zone_index"] = dest_zone_idx
        partner["zone_id"] = order["destination_zone_id"]

        return {
            "order_id": order["order_id"], "assigned_partner": partner["restaurant_id"],
            "pickup_distance_km": round(pickup_sec / 3600 * DELIVERY_SPEED_KMH, 2),
            "delivery_distance_km": round(dropoff_sec / 3600 * DELIVERY_SPEED_KMH, 2),
            "travel_time_sec": round(total_travel_sec, 1), "dispatch_time": current_time,
        }

    def drain_queue(current_time, hour):
        window = cancellation_window_for_hour(hour)
        still_pending = []
        for item in pending_queue:
            if current_time - item["queued_since"] > window:
                dispatch_log.append({
                    "order_id": item["order"]["order_id"], "assigned_partner": None,
                    "pickup_distance_km": None, "delivery_distance_km": None,
                    "travel_time_sec": None, "dispatch_time": item["order"]["timestamp_sec"],
                })
                continue
            result = try_dispatch(item["order"], current_time)
            (still_pending.append(item) if result is None else dispatch_log.append(result))
        pending_queue[:] = still_pending

    while order_stream.has_next():
        if max_orders is not None and count >= max_orders:
            break

        order = order_stream.next()
        current_time = order["timestamp_sec"]
        hour = int(current_time // 3600)

        target_now = HOURLY_ACTIVE_TARGET.get(hour, NORMAL_ACTIVE_PARTNERS)
        release_returned_partners(partners, current_time, target_now)

        if hour != current_hour:
            rebalance_active_partners(partners, HOURLY_ACTIVE_TARGET.get(hour, NORMAL_ACTIVE_PARTNERS), current_time)
            current_hour = hour

        drain_queue(current_time, hour)

        result = try_dispatch(order, current_time)
        (pending_queue.append({"order": order, "queued_since": current_time}) if result is None
         else dispatch_log.append(result))

        count += 1

    for item in pending_queue:
        dispatch_log.append({
            "order_id": item["order"]["order_id"], "assigned_partner": None,
            "pickup_distance_km": None, "delivery_distance_km": None,
            "travel_time_sec": None, "dispatch_time": item["order"]["timestamp_sec"],
        })

    return pd.DataFrame(dispatch_log)

In [ ]:
stream.reset()
for p in partners:
    p["status"] = "available"
    p["return_time"] = 0

full_log = dispatch_greedy(stream, partners, zone_matrix)

# dispatch_greedy already outputs pickup_distance_km and delivery_distance_km separately.
# total_distance_km = full trip per dispatch (pickup leg + delivery leg)
full_log["total_distance_km"] = (
    full_log["pickup_distance_km"].fillna(0) + full_log["delivery_distance_km"].fillna(0)
)

print(full_log["travel_time_sec"].describe())
print("\nNonzero dispatches:", (full_log["travel_time_sec"] > 0).sum(), "/", len(full_log))
print(f"\nAvg pickup distance   (partner→restaurant): {full_log['pickup_distance_km'].mean():.2f} km")
print(f"Avg delivery distance (restaurant→dest):    {full_log['delivery_distance_km'].mean():.2f} km  (target ~2.98 km)")
print(f"Avg total trip distance:                    {full_log['total_distance_km'].mean():.2f} km")

full_log_with_time = full_log.merge(orders_sorted[["order_id", "peak_window"]], on="order_id")
print("\nTravel time by peak_window:\n", full_log_with_time.groupby("peak_window")["travel_time_sec"].describe())


In [ ]:
print("\nSame-zone vs cross-zone dispatches:")
print((full_log["travel_time_sec"] == INTRA_ZONE_SEC).value_counts())

In [ ]:
full_log_zones = full_log.merge(
    orders_sorted[["order_id", "restaurant_id", "destination_zone_id"]], on="order_id"
)
full_log_zones["restaurant_zone_idx"] = full_log_zones["restaurant_id"].map(restaurant_zone_lookup)
full_log_zones["dest_zone_idx"] = full_log_zones["destination_zone_id"].map(zone_id_to_index)
full_log_zones["same_zone"] = full_log_zones["restaurant_zone_idx"] == full_log_zones["dest_zone_idx"]

print(full_log_zones["same_zone"].value_counts())
print(f"\nSame-zone %: {full_log_zones['same_zone'].mean()*100:.2f}%")

full_log_zones = full_log_zones.merge(orders_sorted[["order_id", "peak_window"]], on="order_id")
print(full_log_zones.groupby("peak_window")["same_zone"].apply(lambda s: (~s).sum()))

Day 4

In [ ]:
from collections import deque

WINDOW_SEC = sim_config["batch_window_sec"]  # 120s — matches Build_City_Network and the Smart notebook
DELIVERY_SPEED_KMH = 30  # same speed constant used throughout Day 3

def zone_pair_distance_km(idx1, idx2):
    """Distance in km between two zone indices, using the same time->km conversion as Day 3."""
    sec = INTRA_ZONE_SEC if idx1 == idx2 else zone_matrix[idx1, idx2]
    return sec / 3600 * DELIVERY_SPEED_KMH


dispatch_info = full_log.merge(
    orders_sorted[["order_id", "restaurant_id", "destination_zone_id", "peak_window", "hour"]],
    on="order_id", how="left"
).sort_values("dispatch_time").reset_index(drop=True)

missed_batches = []
recent_by_restaurant = {}
for _, row in dispatch_info.iterrows():
    rid = row["restaurant_id"]
    t = row["dispatch_time"]
    order_id = row["order_id"]
    partner = row["assigned_partner"]
    dest_idx = zone_id_to_index[row["destination_zone_id"]]

    dq = recent_by_restaurant.setdefault(rid, deque())

    # Drop entries older than the WINDOW_SEC window (120s)
    while dq and t - dq[0][1] > WINDOW_SEC:
        dq.popleft()

    # Compare current order against every still-recent order from the same restaurant
    if pd.notna(partner):
        for prev_order_id, prev_t, prev_partner, prev_dest_idx in dq:
            if pd.notna(prev_partner) and prev_partner != partner:
                missed_batches.append({
                    "order_id_1": prev_order_id,
                    "order_id_2": order_id,
                    "restaurant_id": rid,
                    "time_gap_seconds": round(t - prev_t, 1),
                    "distance_between_deliveries_km": round(zone_pair_distance_km(prev_dest_idx, dest_idx), 2),
                })

    dq.append((order_id, t, partner, dest_idx))

missed_batches_df = pd.DataFrame(missed_batches)
print(f"Total missed batch events found: {len(missed_batches_df)}")
missed_batches_df.head(10)

In [ ]:
total_dispatches = len(dispatch_info)
total_missed = len(missed_batches_df)
pct_missed = total_missed / total_dispatches * 100

print(f"Total dispatches: {total_dispatches}")
print(f"Total missed batch events: {total_missed}")
print(f"Missed batches as % of total dispatches: {pct_missed:.2f}%")

print("\nMissed batch time gaps (seconds):\n", missed_batches_df["time_gap_seconds"].describe())
print("\nMissed batch distances (km):\n", missed_batches_df["distance_between_deliveries_km"].describe())

In [ ]:
top_restaurants = (
    missed_batches_df["restaurant_id"]
    .value_counts()
    .head(10)
    .rename_axis("restaurant_id")
    .reset_index(name="missed_batch_count")
)

print("Top 10 restaurants by missed batching failures:")
print(top_restaurants.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

missed_with_hour = missed_batches_df.merge(
    orders_sorted[["order_id", "hour"]],
    left_on="order_id_2", right_on="order_id", how="left"
)

hourly_missed = missed_with_hour.groupby("hour").size().reindex(range(10, 23), fill_value=0)

plt.figure(figsize=(10, 5))
plt.bar(hourly_missed.index, hourly_missed.values, color="tomato", edgecolor="black")
plt.xlabel("Hour of Day")
plt.ylabel("Missed Batch Opportunities")
plt.title("Missed Batch Opportunities by Hour — Greedy Dispatcher")
plt.axvspan(12, 14, color="orange", alpha=0.15, label="Lunch")
plt.axvspan(19, 22, color="purple", alpha=0.15, label="Dinner")
plt.xticks(range(10, 23))
plt.legend()
plt.show()

print(hourly_missed)

In [ ]:
# Check unique restaurants active per peak window, and orders per restaurant per window
window_stats = orders_sorted.groupby("peak_window").agg(
    total_orders=("order_id", "count"),
    unique_restaurants=("restaurant_id", "nunique"),
).reset_index()
window_stats["orders_per_restaurant"] = window_stats["total_orders"] / window_stats["unique_restaurants"]
window_stats["hours_in_window"] = window_stats["peak_window"].map({"lunch": 2, "dinner": 3, "offpeak": 9})
window_stats["orders_per_restaurant_per_hour"] = window_stats["orders_per_restaurant"] / window_stats["hours_in_window"]

print(window_stats)

Day 5

In [ ]:
import os

total_orders     = len(orders_sorted)
total_dispatches = int(full_log["assigned_partner"].notna().sum())

OFFPEAK_HOURS_COUNT = len(HOURLY_ACTIVE_TARGET) - len(PEAK_HOURS) - len(NIGHT_HOURS)

target_partners_weighted_avg = round(
    (NORMAL_ACTIVE_PARTNERS * OFFPEAK_HOURS_COUNT
     + PEAK_ACTIVE_PARTNERS  * len(PEAK_HOURS)
     + NIGHT_ACTIVE_PARTNERS * len(NIGHT_HOURS))
    / (OFFPEAK_HOURS_COUNT + len(PEAK_HOURS) + len(NIGHT_HOURS))
)

total_partners_used = int(full_log["assigned_partner"].nunique())

# total_km_driven = full operational km (pickup leg + delivery leg per dispatch)
# Meaning: sum of partner→restaurant + restaurant→customer for every dispatch
total_km_driven = float(full_log["total_distance_km"].sum())

# avg_delivery_distance_km = restaurant→destination leg ONLY
# Meaning: average distance from restaurant to customer drop-off point
# CDP-anchored target for Hyderabad: ~2.98 km
avg_delivery_distance_km = float(full_log["delivery_distance_km"].mean())

# avg_total_trip_distance_km = partner→restaurant + restaurant→destination
# Meaning: full distance a partner travels per order dispatch
# This is the CDP-comparable metric (CDP reports full trip per order)
avg_total_trip_distance_km = float(full_log["total_distance_km"].mean())

missed_batch_count = len(missed_batches_df)
missed_batch_pct   = missed_batch_count / total_dispatches * 100

baseline_metrics = {
    "total_orders":                 total_orders,
    "total_dispatches":             total_dispatches,
    "total_partners_used":          total_partners_used,
    "target_partners_weighted_avg": target_partners_weighted_avg,
    "total_km_driven":              round(total_km_driven, 2),
    "avg_delivery_distance_km":     round(avg_delivery_distance_km, 2),
    "avg_total_trip_distance_km":   round(avg_total_trip_distance_km, 2),
    "missed_batch_count":           missed_batch_count,
    "missed_batch_pct":             round(missed_batch_pct, 2),
}

print("=" * 55)
print("  GREEDY BASELINE — KEY DERIVED VALUES")
print("=" * 55)
print(f"  Total orders              : {total_orders:,}")
print(f"  Total dispatches          : {total_dispatches:,}")
print(f"  Partners used             : {total_partners_used:,}")
print(f"  Total km driven           : {total_km_driven:,.2f} km")
print(f"    [pickup + delivery legs combined per dispatch]")
print(f"  Avg delivery distance     : {avg_delivery_distance_km:.2f} km")
print(f"    [restaurant → customer drop-off only | CDP target: ~2.98 km]")
print(f"  Avg total trip distance   : {avg_total_trip_distance_km:.2f} km")
print(f"    [partner → restaurant + restaurant → customer | CDP-comparable]")
print(f"  Missed batch count        : {missed_batch_count:,}")
print(f"  Missed batch %            : {missed_batch_pct:.2f}%")
print("=" * 55)


In [ ]:
FIXED_PAYOUT_PER_PARTNER_INR = 40
FUEL_PRICE_PER_LITRE_INR = 111.21  # India national avg petrol price, Goodreturns, retrieved 27 Aug 2026
MILEAGE_KM_PER_LITRE     = 40      # standard 2-wheeler mileage assumption
FUEL_COST_PER_KM_INR     = round(FUEL_PRICE_PER_LITRE_INR / MILEAGE_KM_PER_LITRE, 3)

fixed_component = total_partners_used * FIXED_PAYOUT_PER_PARTNER_INR
fuel_component  = total_km_driven * FUEL_COST_PER_KM_INR
baseline_cost_inr = fixed_component + fuel_component

baseline_metrics["baseline_cost_inr"] = round(baseline_cost_inr, 2)

print(f"  Fuel cost per km : ₹{FUEL_COST_PER_KM_INR}")
print(f"  Fixed component  : ₹{fixed_component:,.2f}  [{total_partners_used} partners × ₹{FIXED_PAYOUT_PER_PARTNER_INR}]")
print(f"  Fuel component   : ₹{fuel_component:,.2f}  [{total_km_driven:,.2f} km × ₹{FUEL_COST_PER_KM_INR}/km]")
print(f"  Total cost       : ₹{baseline_cost_inr:,.2f}")


In [ ]:
import json

os.makedirs("results", exist_ok=True)

# Full dispatch log
full_log.to_csv("results/greedy_dispatch_log.csv", index=False)

# Metrics JSON
with open("results/baseline_metrics.json", "w") as f:
    json.dump(baseline_metrics, f, indent=2)

print("Saved:")
print(" - results/greedy_dispatch_log.csv")
print(" - results/baseline_metrics.json")

In [ ]:
import matplotlib.pyplot as plt

# Partner utilization: fraction of simulation window each partner spent busy delivering
sim_duration_sec = orders_sorted["timestamp_sec"].max() - orders_sorted["timestamp_sec"].min()
partner_busy_time = full_log.groupby("assigned_partner")["travel_time_sec"].sum()
partner_utilization_pct = (partner_busy_time / sim_duration_sec * 100).clip(upper=100)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Orders by hour
orders_by_hour = orders_sorted.groupby("hour").size()
axes[0, 0].bar(orders_by_hour.index, orders_by_hour.values, color="steelblue", edgecolor="black")
axes[0, 0].set_title("Orders by Hour")
axes[0, 0].set_xlabel("Hour of Day")
axes[0, 0].set_ylabel("Order Count")

# KM distribution
axes[0, 1].hist(full_log["delivery_distance_km"], bins=30, color="seagreen", edgecolor="black")
axes[0, 1].set_title("Delivery Distance Distribution (delivery leg only)")
axes[0, 1].set_xlabel("Distance (km)")
axes[0, 1].set_ylabel("Frequency")

# Missed batches by hour
axes[1, 0].bar(hourly_missed.index, hourly_missed.values, color="tomato", edgecolor="black")
axes[1, 0].set_title("Missed Batch Opportunities by Hour")
axes[1, 0].set_xlabel("Hour of Day")
axes[1, 0].set_ylabel("Missed Batch Count")

# Partner utilization %
axes[1, 1].hist(partner_utilization_pct, bins=30, color="darkorange", edgecolor="black")
axes[1, 1].set_title("Partner Utilization %")
axes[1, 1].set_xlabel("Utilization (%)")
axes[1, 1].set_ylabel("Number of Partners")

plt.tight_layout()
plt.savefig("results/baseline_summary.png", dpi=150)
plt.show()

print(f"\nAverage partner utilization: {partner_utilization_pct.mean():.1f}%")
print("Saved: results/baseline_summary.png")

In [ ]:
baseline_paragraph = f"""
Baseline Findings — what the greedy system does and what it costs:

The greedy dispatcher assigns each incoming order to the nearest available delivery
partner in real time, with no coordination across orders. Across a full day of
simulated demand ({total_orders:,} orders), this baseline required {total_partners_used}
active delivery partners to cover {total_dispatches:,} dispatches, driving a combined
{total_km_driven:,.0f} km at an estimated cost of ₹{baseline_cost_inr:,} in partner
payouts. Critically, {missed_batch_count:,} dispatches ({missed_batch_pct:.1f}% of all
deliveries) represent missed batching opportunities — cases where two orders to the
same restaurant within a {WINDOW_SEC}-second window were assigned to different partners purely
because the dispatcher acts greedily on each order in isolation, rather than looking
ahead. These missed batches concentrate heavily during lunch and dinner peaks, when
restaurant-level order density is highest. This baseline is what any intelligent
batching solution must beat: fewer partners, less distance driven, and a lower
missed-batch rate, without violating delivery SLAs.
"""

print(baseline_paragraph)

In [ ]:
TARGET_UNASSIGNED = {"peak": 10.0, "night": 12.0, "normal": 2.0}  # normal midpoint of your 1-3% range

full_log_check = full_log.merge(orders_sorted[["order_id", "hour"]], on="order_id")
full_log_check["is_night"] = full_log_check["hour"].isin(NIGHT_HOURS)
full_log_check["is_peak"] = full_log_check["hour"].isin([12, 13, 19, 20, 21])

print(f"{'Segment':<10}{'Actual %':<12}{'Target %':<12}{'n'}")
for label, mask, target_key in [
    ("peak", full_log_check["is_peak"] & ~full_log_check["is_night"], "peak"),
    ("night", full_log_check["is_night"], "night"),
    ("normal", ~full_log_check["is_peak"] & ~full_log_check["is_night"], "normal"),
]:
    sub = full_log_check[mask]
    actual_pct = sub["assigned_partner"].isna().mean() * 100
    print(f"{label:<10}{actual_pct:<12.1f}{TARGET_UNASSIGNED[target_key]:<12.1f}{len(sub)}")

overall_pct = full_log["assigned_partner"].isna().mean() * 100
print(f"\nOverall unassigned: {overall_pct:.1f}%")